<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [1]</a>'.</span>

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [1]:
import numpy as np
import torch
import tensorly as tl
from tensorly.decomposition import parafac
tl.set_backend('pytorch')
import sparse
import pickle
import os

# -------------------------------
# 1. PyTorch-based BPTF (your version)
# -------------------------------
# Import your PyTorch-based BPTF model (adjust filename as needed)
from own_implementation import BPTF as BPTF_torch

assert os.path.exists('sptensor.pkl'), 'No such file.'
with open('sptensor.pkl', 'rb') as f:
    data = pickle.load(f)
data = data[:, :, :, :12, :]
expected_shape = data.shape
n_components = 10

device = 'cpu'
# Create a tensor from a Poisson distribution (counts) and a matching mask; ensure types match
data_torch = torch.tensor(data, dtype=torch.float64, device=device)
mask_torch = torch.ones(expected_shape, dtype=torch.float64, device=device)

# Instantiate and fit the PyTorch-based BPTF model
model_torch = BPTF_torch(data_shape=expected_shape, n_components=n_components, alpha=0.1, device=device)
model_torch.fit(data_torch, mask=mask_torch, max_iter=50, tol=1e-4, verbose=True)
reconstruction_torch = model_torch.reconstruct(mask=mask_torch, style='arithmetic')
frobenius_diff_torch = torch.norm(data_torch - reconstruction_torch, p='fro').item()
print("PyTorch BPTF reconstruction Frobenius norm difference:", frobenius_diff_torch)

# -------------------------------
# 2. TensorLy CP Decomposition
# -------------------------------
# Use TensorLy's parafac for CP decomposition (same rank as n_components)
cp_decomp = parafac(data_torch, rank=n_components, n_iter_max=100, init='svd')
reconstruction_cp = tl.cp_to_tensor(cp_decomp)
frobenius_diff_cp = torch.norm(data_torch - reconstruction_cp, p='fro').item()
print("TensorLy CP decomposition Frobenius norm difference:", frobenius_diff_cp)

# -------------------------------
# 3. NumPy-based BPTF (Aaron's original implementation)
# -------------------------------
# Import Aaron's BPTF (which uses NumPy/sparse.COO); 
# ensure that the bptf package is in your PYTHONPATH.
from bptf import BPTF as BPTF  # :contentReference[oaicite:2]{index=2}
import bptf

# Create the same data as a NumPy array and a corresponding binary mask
data_np = sparse.COO.from_numpy(data.copy())
mask_np = np.ones(expected_shape, dtype=int)
mask_np = sparse.COO.from_numpy(mask_np.copy())

# def _check_mode(self, m):
#     assert np.isfinite(np.asarray(self.E_DK_M[m])).all()
#     assert np.isfinite(np.asarray(self.G_DK_M[m])).all()
#     assert np.isfinite(np.asarray(self.shp_DK_M[m])).all()
#     assert np.isfinite(np.asarray(self.rte_DK_M[m])).all()

# bptf.BPTF._check_mode = _check_mode

# Instantiate and fit the NumPy-based BPTF model.
# Note: This version uses its own preprocess() function and can work with sparse.COO.
model_np = BPTF(data_shape=data_np.shape, n_components=n_components)
model_np.fit(data_np, mask=mask_np, max_iter=50, verbose=True)

# Reconstruct using arithmetic expectation
reconstruction_np = model_np.reconstruct(mask=mask_np, fill_value=0, drop_diag=False, style='arithmetic')
frobenius_diff_np = np.linalg.norm(data_np - reconstruction_np, ord='fro')
print("NumPy BPTF reconstruction Frobenius norm difference:", frobenius_diff_np)


  0%|                                                                                                       | 0/50 [00:00<?, ?it/s]

ELBO = -162109779.49494576, change = 0.726466151284273, time taken = 7.536017894744873:   0%|               | 0/50 [00:07<?, ?it/s]

ELBO = -162109779.49494576, change = 0.726466151284273, time taken = 7.536017894744873:   2%|▏      | 1/50 [00:07<06:09,  7.54s/it]

ELBO = -455685837.90954065, change = -1.8109706849841711, time taken = 7.980607032775879:   2%|     | 1/50 [00:15<06:09,  7.54s/it]

ELBO = -455685837.90954065, change = -1.8109706849841711, time taken = 7.980607032775879:   4%|▏    | 2/50 [00:15<06:14,  7.80s/it]

...

ELBO = -450321946.3706274, change = -2.289159650720896e-05, time taken = 7.324279546737671:  98%|█▉| 49/50 [06:26<00:07,  7.30s/it]

ELBO = -450321946.3706274, change = -2.289159650720896e-05, time taken = 7.324279546737671: 100%|██| 50/50 [06:26<00:00,  7.31s/it]

ELBO = -450321946.3706274, change = -2.289159650720896e-05, time taken = 7.324279546737671: 100%|██| 50/50 [06:26<00:00,  7.73s/it]

PyTorch BPTF reconstruction Frobenius norm difference: 39365.22307349232


RuntimeError: [enforce fail at alloc_cpu.cpp:117] err == 0. DefaultCPUAllocator: can't allocate memory: you tried to allocate 2832334848000000 bytes. Error code 12 (Cannot allocate memory)